**Import Libraries**

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.preprocessing import LabelEncoder

**Load Clean Dataset**

In [ ]:
DATA = Path("../data/processed")

df = pd.read_csv(DATA/"cleaned_dataset.csv")

print(df.shape)

df.head()

**Removal Invalid Target Values**

In [ ]:
df = df[
    df["Flag_Invalid_VO2"] == False
].copy()

print(df.shape)

**Remove Missing Target**

In [ ]:
df = df.dropna(
    subset=["VO2max"]
)

print(df.shape)

**Encode Sex**

In [ ]:
encoder = LabelEncoder()

df["Sex"] = encoder.fit_transform(
    df["Sex"]
)

**x_ant (Anthropometric Features)**

In [ ]:
df["BMI_Age"] = (
    df["BMI"] *
    df["Age"]
)
df["Height_Weight_Ratio"] = (
    df["Height_cm"] /
    df["Weight_kg"]
)
df["Body_Surface_Area"] = np.sqrt(
    (
        df["Height_cm"] *
        df["Weight_kg"]
    ) / 3600
)
df["BMI_Squared"] = (
    df["BMI"] ** 2
)
df["Age_Squared"] = (
    df["Age"] ** 2
)

**x_resp (Cardiovascular Response)**

In [ ]:
df["Predicted_HRmax"] = (
    220 -
    df["Age"]
)
df["Relative_HR_Response"] = (
    df["Max_HR"] /
    df["Predicted_HRmax"]
)
df["HR_Difference"] = (
    df["Predicted_HRmax"] -
    df["Max_HR"]
)
exercise_cols = [
    "Warmup_Time",
    "Stage1_Time",
    "Stage2_Time"
]

df["Exercise_Duration"] = (
    df[exercise_cols]
    .fillna(0)
    .sum(axis=1)
)
df["VO2_per_HR"] = (
    df["VO2max"] /
    df["Max_HR"]
)

**x_met (Metabolic Features)**

In [ ]:
df["Absolute_VO2"] = (
    df["VO2max"] *
    df["Weight_kg"]
) / 1000
df["Weight_per_Height"] = (
    df["Weight_kg"] /
    df["Height_cm"]
)
bmi_map = {
    "Underweight":0,
    "Normal":1,
    "Overweight":2,
    "Obese":3
}

df["BMI_Category_Code"] = (
    df["BMI_Category"]
    .map(bmi_map)
)


**Interaction Features**

In [ ]:
df["Age_BMI"] = (
    df["Age"] *
    df["BMI"]
)
df["Weight_Height"] = (
    df["Weight_kg"] *
    df["Height_cm"]
)
df["HR_BMI"] = (
    df["Max_HR"] *
    df["BMI"]
)
df["HR_Age"] = (
    df["Max_HR"] *
    df["Age"]
)

**Physiological Sanity Checks**

In [ ]:
df = df[
    df["Relative_HR_Response"] > 0
]

df = df[
    df["Relative_HR_Response"] < 2
]

**Feature Groups**

In [ ]:
x_ant = [
    "Age",
    "Sex",
    "Height_cm",
    "Weight_kg",
    "BMI",
    "BMI_Age",
    "Body_Surface_Area",
    "BMI_Squared",
    "Age_Squared",
    "Height_Weight_Ratio"
]
x_resp = [
    "Max_HR",
    "Predicted_HRmax",
    "Relative_HR_Response",
    "HR_Difference",
    "Exercise_Duration"
]
x_met = [
    "Weight_per_Height",
    "BMI_Category_Code"
]


**Final Feature Matrix**

In [ ]:
feature_columns = (
    x_ant +
    x_resp +
    x_met
)

X = df[feature_columns]

y = df["VO2max"]

**Verify Missing Values**

In [ ]:
X.isnull().sum()

**Feature Correlation Matrix**

In [ ]:
corr = X.corr().abs()

upper = corr.where(
    np.triu(
        np.ones(corr.shape),
        k=1
    ).astype(bool)
)

high_corr = [
    column
    for column in upper.columns
    if any(
        upper[column] > 0.95
    )
]

print(high_corr)

**Save Feature Dataset**

In [ ]:
feature_dataset = pd.concat(
    [
        X,
        y
    ],
    axis=1
)

feature_dataset.to_csv(
    DATA/"final_features.csv",
    index=False
)

print("Saved successfully.")